## Task 1.2: Key Assumptions

**Paper:** *Finding Deceptive Opinion Spam by Any Stretch of the Imagination*  
**Authors:** Myle Ott, Yejin Choi, Claire Cardie, Jeffrey T. Hancock  
**Venue:** ACL 2011  

---

## Key Assumptions

---

### Assumption 1

**Assumption:**  
The dataset is perfectly balanced — exactly 400 deceptive and 400 truthful reviews — and the classifier is designed and evaluated under this 50/50 class distribution.

**Why the method needs it:**  
The Naïve Bayes formulation used in the paper simplifies from the full Bayes rule (Equation 1) to the maximum likelihood classifier (Equation 2) specifically because the class prior Pr(y=c) is uniform — which is only true when both classes are equally represented. If the classes were imbalanced, the prior would no longer cancel out, and the simplified classifier would produce systematically biased predictions without any modification.

**Violation scenario:**  
In a real-world hotel review platform such as TripAdvisor or Google Reviews, fake reviews are estimated to represent only 10–20% of total reviews — far from the 50% assumed here. A classifier trained and evaluated on the balanced dataset would encounter a very different class distribution in deployment, causing it to over-predict the minority (deceptive) class and producing unacceptably high false positive rates that would flag genuine customers as spammers.

**Paper reference:**  
Section 3.2 (Truthful Opinions from TripAdvisor) — the authors explicitly balance the dataset by selecting exactly 400 truthful reviews to match the 400 deceptive reviews. Section 4.4 (Classifiers), Equation 1 and Equation 2 — the simplification from full Bayes to maximum likelihood is justified explicitly by the balanced class assumption.

---

### Assumption 2

**Assumption:**  
Deceptive reviews written by anonymous Amazon Mechanical Turk (AMT) crowdworkers are representative of real-world deceptive opinion spam produced by professional paid reviewers or coordinated spam campaigns.

**Why the method needs it:**  
The entire supervised learning pipeline depends on the training labels being a valid proxy for the real-world phenomenon being detected. If AMT Turkers write deceptive reviews that are linguistically different from professionally written spam — for example, because Turkers are less motivated, less skilled at mimicking genuine writing, or write shorter reviews — then the n-gram and LIWC features the SVM learns will not generalise to the actual spam that appears on review platforms. The classifier would be learning to detect "amateur fake reviews" rather than "deceptive opinion spam" as it exists in practice.

**Violation scenario:**  
A professional reputation management company that trains its employees with templates, instructs them to vary sentence structure, and coaches them on avoiding detectable patterns would produce spam that looks linguistically very different from what AMT Turkers write. A model trained on AMT data would fail systematically against such professionally crafted deceptive reviews, as the distributional patterns it learned would not match the adversarially optimised writing style of professional spammers.

**Paper reference:**  
Section 3.1 (Deceptive Opinions via Mechanical Turk) — the authors describe the AMT data collection process and acknowledge that Turkers were paid one dollar per review and given 30 minutes maximum. Section 6 (Conclusion and Future Work) — the authors explicitly identify extending evaluation to "other domains" as future work, implicitly acknowledging the generalisation limitation of the AMT-sourced deceptive reviews.

---

### Assumption 3

**Assumption:**  
Deceptive writing produces stable, detectable surface-level linguistic signals — specifically, that deceptive reviews will consistently use more imaginative, emotionally loaded, and spatially vague language compared to truthful reviews, and that these signals are learnable from a bag-of-words n-gram representation.

**Why the method needs it:**  
The entire text categorization approach (Section 4.3) relies on n-gram frequency features being discriminative between deceptive and truthful classes. This only works if deceptive writing systematically differs from truthful writing at the word and phrase level in a consistent, learnable way. If deceptive reviewers used vocabulary and phrasing that was statistically indistinguishable from truthful reviewers — for example, by deliberately copying the style and specific details of genuine reviews — then no word-frequency–based classifier could perform better than chance, regardless of model capacity.

**Violation scenario:**  
Consider a spam campaign in which the spammer first scrapes a large set of genuine reviews for a hotel, studies the vocabulary distribution, and then generates deceptive reviews by paraphrasing authentic reviews with minor modifications. Such a strategy would deliberately match the unigram and bigram distribution of truthful reviews, collapsing the feature space separation that the SVM's hyperplane depends on. This scenario is particularly realistic given the rise of LLM-generated fake reviews, which can produce text that is stylistically indistinguishable from human writing at the surface level.

**Paper reference:**  
Section 4.3 (Text Categorization) — the paper's core hypothesis that n-gram features are discriminative. Section 5 (Results and Discussion), Table 5 — the specific discriminative features (e.g., "bathroom", "location" for truthful; "husband", "vacation" for deceptive) demonstrate that this assumption holds for the AMT dataset, but these are domain-specific signals that may not persist under adversarial conditions.

---

### Assumption 4

**Assumption:**  
The classification task is restricted to positive (5-star) hotel reviews only, and the learned deception signals are assumed to generalise within this narrow positive-sentiment, single-domain setting.

**Why the method needs it:**  
The paper's feature analysis reveals that some of the strongest truthful/deceptive discriminative features are sentiment-bearing words (e.g., deceptive reviews show more positive emotion words and fewer negative emotion terms — Table 5). This pattern only holds because all reviews in the dataset are positive 5-star reviews. The deception signal the classifier learns is entangled with the sentiment signal of the reviews. If the model were applied to negative reviews, the sentiment-related features would flip, and the learned weights would produce incorrect predictions.

**Violation scenario:**  
A competitor might post fake negative reviews ("review bombing") to damage a hotel's reputation — a form of deceptive opinion spam explicitly acknowledged by the authors in footnote 4. A classifier trained only on positive deceptive reviews would have no valid signal for detecting negative deceptive reviews, as the discriminative features it learned (such as increased positive emotion terms being a deception signal) would be completely reversed in sentiment direction.

**Paper reference:**  
Section 3.2 (Truthful Opinions from TripAdvisor) — the authors explicitly filter to 5-star reviews only. Footnote 4 (Section 1, Introduction) — the authors acknowledge that negative deceptive spam exists. Section 6 (Conclusion and Future Work) — extending to negative opinions is explicitly listed as the primary direction for future work, confirming that this restriction is a known limitation of the current method.